In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
import pandas as pd
import ast
import re

df = pd.read_csv("train.csv")

def fix_dialog(s):
    s = s.strip()
    s = re.sub(r"'\s+'", "', '", s)  
    return ast.literal_eval(s)

df['dialog'] = df['dialog'].apply(fix_dialog)

def fix_num_list(s):
    s = re.sub(r'\s+', ', ', s.strip())
    return ast.literal_eval(s)

df['act'] = df['act'].apply(fix_num_list)
df['emotion'] = df['emotion'].apply(fix_num_list)


data = []
for dialog in df['dialog']:
    for i in range(len(dialog)-1):
        context = dialog[i]
        response = dialog[i+1]
        data.append((context, response))

print("Pairs:", len(data))
data[:5]

data = data[:5000]   # ~7% of dataset for faster training because of lack or resources


In [ ]:
# Add special tokens
special_tokens = ["<PAD>", "<SOS>", "<EOS>"]
vocab = list(set(" ".join([f"{c} {r}" for c, r in data]).replace("!", "").split()))
vocab = special_tokens + vocab
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)
embed_dim, hidden_dim, latent_dim = 32, 64, 16


In [ ]:

# Encoder

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        emb = self.embedding(x)
        # Assuming single layer GRU
        _, h = self.rnn(emb)
        mu = self.mu(h.squeeze(0))
        logvar = self.logvar(h.squeeze(0))
        return mu, logvar



In [ ]:

# Decoder

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, z):
        emb = self.embedding(x)
        h0 = self.latent_to_hidden(z).unsqueeze(0)
        output, _ = self.rnn(emb, h0)
        logits = self.out(output)
        return logits

In [ ]:

# CVAE Model (Updated with BOW MLP)

class CVAE(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_dim, latent_dim)
        self.vocab_size = vocab_size

        self.mlp_bow = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size)
        ).to(device)


    def forward(self, x, y):
        mu, logvar = self.encoder(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std

        if y.size(1) < 2:
            # Handle minimal response length for decoder input
            return None, mu, logvar, None

        logits = self.decoder(y[:, :-1], z)

        # BOW Prediction
        bow_logits = self.mlp_bow(z)

        return logits, mu, logvar, bow_logits

    @torch.no_grad()
    def generate_diversity(self, x_context, num_samples=3, max_len=8):
        """Generates multiple diverse responses by sampling 'z' multiple times."""
        self.eval()

        # Calculate mu, logvar once from the context
        mu, logvar = self.encoder(x_context.to(device))

        responses = []
        for _ in range(num_samples):
            # 1. Sample latent variable z from the posterior
            std = torch.exp(0.5 * logvar)
            z = mu + torch.randn_like(mu) * std

            # 2. Sequential Decoding (Conditioned on z)
            inp = torch.tensor([[word2idx["<SOS>"]]], dtype=torch.long).to(device)
            outputs = []

            for _ in range(max_len):
                logits = self.decoder(inp, z)
                probs = F.softmax(logits[:, -1, :], dim=-1)
                # Sample the next token (instead of greedy argmax, for more natural diversity)
                next_token = torch.multinomial(probs, num_samples=1)
                next_token_id = next_token.item()

                if next_token_id == word2idx["<EOS>"]:
                    break

                # Exclude <PAD> token from output
                if next_token_id == word2idx["<PAD>"]:
                    break

                outputs.append(next_token_id)
                inp = torch.cat([inp, next_token], dim=1)

            responses.append(" ".join([idx2word[i] for i in outputs]))

        self.train()
        return responses


In [ ]:
def sentence_to_tensor(sentence, add_sos=False):
    # Use <PAD> for OOV words
    tokens = [word2idx.get(w, word2idx["<PAD>"]) for w in sentence.split()]
    if add_sos:
        tokens = [word2idx["<SOS>"]] + tokens
    tokens.append(word2idx["<EOS>"])
    return torch.tensor([tokens], dtype=torch.long).to(device)

def get_bow_target(y_tensor, vocab_size):
    
    # y_tensor is (1, seq_len)
    bow_target = torch.zeros(y_tensor.size(0), vocab_size).to(device)

    # Get all unique words in the response (excluding special tokens)
    special_indices = [word2idx[t] for t in ["<SOS>", "<EOS>", "<PAD>"]]
    unique_words = [idx for idx in y_tensor.flatten().unique().tolist() if idx not in special_indices]

    # Set the corresponding indices in the target vector to 1.0
    if unique_words:
        bow_target[0].scatter_(0, torch.tensor(unique_words).to(device), 1.0)

    return bow_target

In [ ]:
import time

print(f"Using device: {device}")
model = CVAE(vocab_size, embed_dim, hidden_dim, latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10

# Loss Weights
KL_WEIGHT = 0.01
BOW_WEIGHT = 0.1

print(f"Starting training on {len(data)} pairs for {epochs} epochs...\n")

start_time = time.time()

for epoch in range(epochs):
    epoch_start = time.time()

    total_loss = 0
    total_recon_loss = 0
    total_kl_loss = 0
    total_bow_loss = 0

    for i, (context, response) in enumerate(data):
        x = sentence_to_tensor(context)
        y = sentence_to_tensor(response, add_sos=True)

        result = model(x, y)
        if result[0] is None:
            continue

        logits, mu, logvar, bow_logits = result

        # 1) Reconstruction Loss
        recon_loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            y[:, 1:].reshape(-1)
        )

        # 2) KL Loss
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

        # 3) BOW Loss
        bow_target = get_bow_target(y, vocab_size)
        bow_loss = F.binary_cross_entropy_with_logits(bow_logits, bow_target)

        # Total Loss
        loss = recon_loss + KL_WEIGHT * kl_loss + BOW_WEIGHT * bow_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon_loss += recon_loss.item()
        total_kl_loss += kl_loss.item() * KL_WEIGHT
        total_bow_loss += bow_loss.item() * BOW_WEIGHT

        
        if (i + 1) % 500 == 0:
            print(f"  Batch {i+1}/{len(data)}")

    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Total Loss: {total_loss:.4f} | "
          f"Recon: {total_recon_loss:.4f} | "
          f"KL: {total_kl_loss:.4f} | "
          f"BOW: {total_bow_loss:.4f} | "
          f"Time: {epoch_time:.2f}s")

total_time = time.time() - start_time
print(f"\n✅ Training finished in {total_time:.2f}s")


In [ ]:

# Generate Diverse Responses (Demonstrates CVAE's core function)

print("\n" + "="*50)
test_context = "what are your hobbies?"
x_test = sentence_to_tensor(test_context)
num_responses = 3

print(f"Input Context: '{test_context}'")
print(f"Generating {num_responses} diverse responses by sampling Z:")

diverse_responses = model.generate_diversity(x_test, num_samples=num_responses)

for i, resp in enumerate(diverse_responses):
    print(f"Response {i+1}: {resp}")

print("\n" + "="*50)
test_context = "How are you?"
x_test = sentence_to_tensor(test_context)

print(f"Input Context: '{test_context}'")
print(f"Generating {num_responses} diverse responses by sampling Z:")

diverse_responses = model.generate_diversity(x_test, num_samples=num_responses)

for i, resp in enumerate(diverse_responses):
    print(f"Response {i+1}: {resp}")
print("="*50)